In [105]:
import numpy as np
from scipy.stats import multivariate_normal
from scipy.special import logsumexp

def approximate_mutual_information(voltage_matrix, N=100, num_samples=10000):
    """
    Approximate I(A; B) via Gaussian mixture and Monte Carlo sampling.
    
    Parameters:
    - voltage_matrix: (n, K) array of voltages V_i(u)
    - N: number of particles
    - num_samples: number of Monte Carlo samples
    
    Returns:
    - Approximation of I(A; B)
    """
    n, K = voltage_matrix.shape
    V = voltage_matrix

    # confine V to be between 1./N and 1-1./N to avoid numerical issues due tto gaussian approx of binomial
    V = np.clip(V, 1./N, 1- 1./N)
    
    # Compute means and covariances for each node
    means = N * V
    covs = np.array([
        np.diag(N * V[u] * (1 - V[u]))
        for u in range(n)
    ])

    # Compute conditional entropies H(A|B=u)
    H_cond_list = []
    log_2pi_e = np.log(2 * np.pi * np.e)
    for u in range(n):
        log_det = np.sum(np.log(N * V[u] * (1 - V[u])))
        #print(f"log_det for u={u}: {log_det}")  # Debugging output
        H_u = 0.5 * (K * log_2pi_e + log_det)
        H_cond_list.append(H_u)
    H_cond = np.mean(H_cond_list)

    # Monte Carlo sampling to estimate H(A)
    np.random.seed(42)  # For reproducibility
    num_samples = min(num_samples, 1000000)  # Limit to avoid excessive memory usage
    if num_samples < 1:
        raise ValueError("num_samples must be at least 1")
    us = np.random.randint(n, size=num_samples)
    samples = np.array([np.random.multivariate_normal(means[u], covs[u]) for u in us])    
    
    # Estimate log-density at samples under mixture
    inv_covs = 1.0 / np.array([np.diag(cov) for cov in covs])  # (n, K)
    log_det_covs = np.sum(np.log([np.diag(cov) for cov in covs]), axis=1)  # (n,)
    
    log_probs = []
    for x in samples:
        diffs = means - x  # (n, K)
        mahal = np.sum(diffs**2 * inv_covs, axis=1)  # (n,)
        logpdfs = -0.5 * (K * np.log(2 * np.pi) + log_det_covs + mahal)
        logpdfs -= np.log(n)
        log_p_x = logsumexp(logpdfs)
        log_probs.append(log_p_x)
    H_marginal = -np.mean(log_probs)
    

    # Mutual information
    I_est = H_marginal - H_cond
    return I_est, H_marginal, H_cond, H_cond_list   


In [106]:
%%time

# Example usage

n = 100   # number of nodes
K = 3     # number of sources
N = 1000  # number of particles per source

# Random example voltages between 0 and 1 (replace with your data)
voltage_matrix = np.array([np.arange(0,1,1/n), np.arange(1,0,-1/n),\
     np.concatenate([np.arange(0,1,2/n), np.arange(1,0,-2/n)])]).T
voltage_matrix.T


# Estimate mutual information
I_est , H_marginal, H_cond, h_cond_list= approximate_mutual_information(voltage_matrix, N,num_samples=1000)
print(f"Estimated I(A; B) ≈ {I_est:.4f} nats, h_marginal ≈ {H_marginal:.4f} nats, h_cond ≈ {H_cond:.4f} nats")



Estimated I(A; B) ≈ 3.8163 nats, h_marginal ≈ 15.4186 nats, h_cond ≈ 11.6023 nats
CPU times: user 77.9 ms, sys: 3.13 ms, total: 81 ms
Wall time: 96.2 ms


In [119]:


# Estimate mutual information
for i in range(2,10):

    voltage_matrix=np.eye(i)  # Example with i nodes and i sources
    #print(f"voltage_matrix=\n{voltage_matrix}\n ")
    # Estimate mutual information
    I_est , H_marginal, H_cond, H_cond_list= approximate_mutual_information(voltage_matrix,N=1000000,num_samples=100000)
    print(f"Estimated I(A; B) ≈ {I_est:.4f} nats, h_marginal ≈ {H_marginal:.4f} nats, h_cond ≈ {H_cond:.4f} nats")
    print(f"log({i})-I_est = {np.log(i) - I_est:.4f}")


Estimated I(A; B) ≈ 0.6921 nats, h_marginal ≈ 3.5300 nats, h_cond ≈ 2.8379 nats
log(2)-I_est = 0.0010
Estimated I(A; B) ≈ 1.0989 nats, h_marginal ≈ 5.3557 nats, h_cond ≈ 4.2568 nats
log(3)-I_est = -0.0003
Estimated I(A; B) ≈ 1.3890 nats, h_marginal ≈ 7.0647 nats, h_cond ≈ 5.6758 nats
log(4)-I_est = -0.0027
Estimated I(A; B) ≈ 1.6144 nats, h_marginal ≈ 8.7091 nats, h_cond ≈ 7.0947 nats
log(5)-I_est = -0.0049
Estimated I(A; B) ≈ 1.7967 nats, h_marginal ≈ 10.3103 nats, h_cond ≈ 8.5136 nats
log(6)-I_est = -0.0050
Estimated I(A; B) ≈ 1.9454 nats, h_marginal ≈ 11.8780 nats, h_cond ≈ 9.9326 nats
log(7)-I_est = 0.0005
Estimated I(A; B) ≈ 2.0808 nats, h_marginal ≈ 13.4323 nats, h_cond ≈ 11.3515 nats
log(8)-I_est = -0.0014
Estimated I(A; B) ≈ 2.1936 nats, h_marginal ≈ 14.9641 nats, h_cond ≈ 12.7704 nats
log(9)-I_est = 0.0036


In [90]:
np.exp(0.955)

2.598670582919522

In [ ]:
%cd /Users/yoavfreund/projects/VoltageDimentionalReduction/clean_code

%pwd

import numpy as np
from typing import Union, Optional, List, Any, Tuple, Callable, Dict
from itertools import product
import pandas
import matplotlib.pyplot as plt

from scipy.sparse.linalg import cg
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import lil_matrix, csr_matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from sklearn.manifold import MDS
from sklearn.datasets import fetch_openml


import landmark
import voltagemap
import problem
import solver
import visualization
import setofpoints
import kmeans
from Utilities import config



workspace_file="../../Voltage_Temp/Intermediates/workspace.pkl"
!ls -l $workspace_file
import dill
with open(workspace_file, "rb") as f:
    workspace_data = dill.load(f)

voltage_map=workspace_data["voltage_map"].all_solutions()
voltage_map.shape




/Users/yoavfreund/projects/VoltageDimentionalReduction/clean_code
-rw-r--r--@ 1 yoavfreund  staff  11298377 Jul 10 11:01 ../../Voltage_Temp/Intermediates/workspace.pkl


(1000, 13)

In [20]:
voltage_map.shape

(1000, 13)

In [21]:
voltage_map.shape

(1000, 13)

In [30]:
%%time

# Example usage

n = 1000   # number of nodes
K = 1      # number of sources
N = 10   # number of particles per source


# Estimate mutual information
I_est , H_marginal, H_cond= approximate_mutual_information(voltage_map[:,0:1], N,num_samples=1000)
print(f"Estimated I(A; B) ≈ {I_est:.4f} nats")


Estimated I(A; B) ≈ 0.2496 nats
CPU times: user 74.8 ms, sys: 3.54 ms, total: 78.3 ms
Wall time: 78.9 ms


In [31]:
for K in range(1, 13):
    I_est , H_marginal, H_cond= approximate_mutual_information(voltage_map[:,0:K], N,num_samples=1000)
    print(f"Estimated I(A; B) ≈ {I_est:.4f} nats, h_marginal ≈ {H_marginal:.4f} nats, h_cond ≈ {H_cond:.4f} nats")

Estimated I(A; B) ≈ 0.2496 nats, h_marginal ≈ -1.9682 nats, h_cond ≈ -2.2179 nats
Estimated I(A; B) ≈ 0.6606 nats, h_marginal ≈ -1.0599 nats, h_cond ≈ -1.7205 nats
Estimated I(A; B) ≈ 1.0323 nats, h_marginal ≈ -0.2436 nats, h_cond ≈ -1.2759 nats
Estimated I(A; B) ≈ 1.3318 nats, h_marginal ≈ 0.1612 nats, h_cond ≈ -1.1706 nats
Estimated I(A; B) ≈ 1.6221 nats, h_marginal ≈ 0.2638 nats, h_cond ≈ -1.3583 nats
Estimated I(A; B) ≈ 1.7810 nats, h_marginal ≈ 0.7826 nats, h_cond ≈ -0.9984 nats
Estimated I(A; B) ≈ 1.9488 nats, h_marginal ≈ 1.2256 nats, h_cond ≈ -0.7232 nats
Estimated I(A; B) ≈ 2.1878 nats, h_marginal ≈ 1.2367 nats, h_cond ≈ -0.9512 nats
Estimated I(A; B) ≈ 2.3529 nats, h_marginal ≈ 1.7473 nats, h_cond ≈ -0.6056 nats
Estimated I(A; B) ≈ 2.4236 nats, h_marginal ≈ 1.9048 nats, h_cond ≈ -0.5188 nats
Estimated I(A; B) ≈ 2.4787 nats, h_marginal ≈ 2.4110 nats, h_cond ≈ -0.0676 nats
Estimated I(A; B) ≈ 2.5485 nats, h_marginal ≈ 3.2814 nats, h_cond ≈ 0.7328 nats


In [17]:
Estimated I(A; B) ≈ 4.2517 nats
Estimated I(A; B) ≈ 4.0096 nats
Estimated I(A; B) ≈ 3.8795 nats

6.907755278982137